# 04 - Datasets, DataLoader & Transforms

## Learning Objectives

- Understand the `torch.utils.data.Dataset` protocol (`__len__` and `__getitem__`)
- Create custom Dataset classes from numpy arrays and CSV-like data
- Use `DataLoader` for batching, shuffling, and parallel data loading
- Build transform pipelines with `torchvision.transforms`
- Understand `collate_fn` for variable-length data
- Split data into train/val/test with `random_split`

## Prerequisites

- Notebooks 01-03 of this series
- Basic Python (classes, `__len__`, `__getitem__`)
- PyTorch installed (`pip install torch torchvision`)

## Table of Contents

1. [The Dataset Protocol](#1-the-dataset-protocol)
2. [Creating a Custom Dataset from NumPy Arrays](#2-creating-a-custom-dataset-from-numpy-arrays)
3. [TensorDataset: The Quick Path](#3-tensordataset-the-quick-path)
4. [DataLoader: Batching, Shuffling, Workers](#4-dataloader-batching-shuffling-workers)
5. [Transforms Pipeline](#5-transforms-pipeline)
6. [collate_fn for Variable-Length Data](#6-collate_fn-for-variable-length-data)
7. [random_split for Train/Val/Test](#7-random_split-for-trainvaltest)
8. [Exercises](#8-exercises)
9. [Common Mistakes & Debugging Tips](#9-common-mistakes--debugging-tips)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")

## 1. The Dataset Protocol

A PyTorch `Dataset` is any object that implements two methods:

| Method | Purpose |
|---|---|
| `__len__(self)` | Return the total number of samples |
| `__getitem__(self, idx)` | Return the sample (and optionally label) at index `idx` |

This is the same protocol that lets `DataLoader` iterate over your data in batches.

In [ ]:
# Minimal Dataset example
class SimpleDataset(Dataset):
    """A toy dataset that returns (x, x^2) pairs."""

    def __init__(self, size=100):
        self.x = torch.linspace(0, 1, size)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        x = self.x[idx]
        y = x ** 2
        return x, y


ds = SimpleDataset(size=10)
print(f"Dataset size: {len(ds)}")
print(f"First sample: {ds[0]}")
print(f"Fifth sample: {ds[4]}")

## 2. Creating a Custom Dataset from NumPy Arrays

A very common pattern: you have features and labels as NumPy arrays and want to wrap them for PyTorch.

In [ ]:
class NumpyDataset(Dataset):
    """Wraps NumPy arrays as a PyTorch Dataset."""

    def __init__(self, X, y):
        # Convert to tensors once (not on every __getitem__ call)
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Simulate some data
np.random.seed(42)
X_np = np.random.randn(200, 5).astype(np.float32)  # 200 samples, 5 features
y_np = np.random.randint(0, 3, size=200)             # 3 classes

dataset = NumpyDataset(X_np, y_np)
print(f"Dataset size: {len(dataset)}")

sample_x, sample_y = dataset[0]
print(f"Sample feature shape: {sample_x.shape}")
print(f"Sample label: {sample_y}")
print(f"Feature dtype: {sample_x.dtype}")
print(f"Label dtype: {sample_y.dtype}")

In [ ]:
# Dataset with on-the-fly transforms
class TransformedDataset(Dataset):
    """Dataset that applies a transform to each sample."""

    def __init__(self, X, y, transform=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        if self.transform:
            x = self.transform(x)
        return x, y


# Example: a simple normalization transform
def normalize(x):
    return (x - x.mean()) / (x.std() + 1e-8)

ds_transformed = TransformedDataset(X_np, y_np, transform=normalize)

x_raw, _ = NumpyDataset(X_np, y_np)[0]
x_norm, _ = ds_transformed[0]
print(f"Raw features:        {x_raw}")
print(f"Normalized features: {x_norm}")
print(f"Normalized mean: {x_norm.mean():.6f}, std: {x_norm.std():.6f}")

## 3. TensorDataset: The Quick Path

If your data is already in tensors, `TensorDataset` saves boilerplate.

In [ ]:
# TensorDataset wraps tensors -- each tensor becomes one "column"
X_tensor = torch.randn(100, 5)
y_tensor = torch.randint(0, 3, (100,))

tensor_ds = TensorDataset(X_tensor, y_tensor)

print(f"Size: {len(tensor_ds)}")
print(f"Sample: {tensor_ds[0]}")

# Equivalent to:
# class ManualTensorDataset(Dataset):
#     def __init__(self, X, y):
#         self.X, self.y = X, y
#     def __len__(self):
#         return len(self.X)
#     def __getitem__(self, idx):
#         return self.X[idx], self.y[idx]

## 4. DataLoader: Batching, Shuffling, Workers

`DataLoader` wraps a `Dataset` and provides:

| Parameter | Description | Typical Value |
|---|---|---|
| `batch_size` | Number of samples per batch | 32, 64, 128 |
| `shuffle` | Randomize order each epoch | `True` for train, `False` for val/test |
| `num_workers` | Parallel data loading processes | 0 (main process), 2-8 for large datasets |
| `drop_last` | Drop incomplete final batch | `True` for training (consistent batch sizes) |
| `pin_memory` | Faster CPU-to-GPU transfer | `True` when using CUDA |

In [ ]:
# Basic DataLoader usage
torch.manual_seed(42)

X = torch.randn(100, 5)
y = torch.randint(0, 3, (100,))
ds = TensorDataset(X, y)

loader = DataLoader(
    ds,
    batch_size=16,
    shuffle=True,
    num_workers=0,   # 0 = load in main process
    drop_last=False,
)

print(f"Dataset size: {len(ds)}")
print(f"Number of batches: {len(loader)}")
print(f"Batch size: {loader.batch_size}")

In [ ]:
# Iterate over batches
for batch_idx, (X_batch, y_batch) in enumerate(loader):
    print(f"Batch {batch_idx}: X shape={X_batch.shape}, y shape={y_batch.shape}")
    if batch_idx >= 3:
        print("...")
        break

In [ ]:
# drop_last=True: useful when batch norm or other layers need consistent batch sizes
loader_drop = DataLoader(ds, batch_size=16, shuffle=True, drop_last=True)

print(f"With drop_last=False: {len(DataLoader(ds, batch_size=16))} batches")
print(f"  Last batch size: {100 % 16} samples")
print(f"With drop_last=True:  {len(loader_drop)} batches")
print(f"  All batches have exactly 16 samples")

In [ ]:
# Full training loop pattern with DataLoader
torch.manual_seed(42)

# Synthetic data
X_train = torch.randn(500, 10)
y_train = torch.randint(0, 3, (500,))
train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

# Simple model
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 3),
)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(5):
    model.train()
    epoch_loss = 0.0
    n_batches = 0

    for X_batch, y_batch in train_loader:
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    print(f"Epoch {epoch+1}: avg loss = {avg_loss:.4f}")

## 5. Transforms Pipeline

**`torchvision.transforms`** provides composable data transformations, commonly used for image preprocessing.

Key transforms:

| Transform | Description |
|---|---|
| `ToTensor()` | Convert PIL Image / ndarray to tensor, scale to [0, 1] |
| `Normalize(mean, std)` | Per-channel normalization |
| `Resize(size)` | Resize image |
| `RandomHorizontalFlip()` | Data augmentation |
| `Compose([...])` | Chain multiple transforms |

In [ ]:
try:
    from torchvision import transforms
    HAS_TORCHVISION = True
except ImportError:
    HAS_TORCHVISION = False
    print("torchvision not installed. Install with: pip install torchvision")
    print("Skipping torchvision examples (concepts still explained).")

if HAS_TORCHVISION:
    print("torchvision available.")

In [ ]:
# Compose: chain transforms together
if HAS_TORCHVISION:
    # Typical image preprocessing pipeline
    train_transform = transforms.Compose([
        transforms.ToTensor(),                        # HWC uint8 -> CHW float [0,1]
        transforms.Normalize(                         # per-channel normalization
            mean=[0.485, 0.456, 0.406],               # ImageNet mean
            std=[0.229, 0.224, 0.225],                # ImageNet std
        ),
    ])

    # Simulate an image as a numpy array (H=32, W=32, C=3, uint8)
    np.random.seed(42)
    fake_image = np.random.randint(0, 256, (32, 32, 3), dtype=np.uint8)

    # Apply transform
    tensor_image = train_transform(fake_image)
    print(f"Before transform: shape={fake_image.shape}, dtype={fake_image.dtype}")
    print(f"After transform:  shape={tensor_image.shape}, dtype={tensor_image.dtype}")
    print(f"Value range: [{tensor_image.min():.3f}, {tensor_image.max():.3f}]")
else:
    print("Skipped: torchvision not available.")

In [ ]:
# Data augmentation transforms (training only)
if HAS_TORCHVISION:
    train_augment = transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    # Validation/test: no augmentation, just normalization
    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    print("Training transform (with augmentation):")
    print(train_augment)
    print("\nValidation transform (no augmentation):")
    print(val_transform)
else:
    print("Skipped: torchvision not available.")

In [ ]:
# Custom transform as a callable class
class AddGaussianNoise:
    """Add Gaussian noise to a tensor (data augmentation)."""

    def __init__(self, mean=0.0, std=0.1):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        noise = torch.randn_like(tensor) * self.std + self.mean
        return tensor + noise

    def __repr__(self):
        return f"AddGaussianNoise(mean={self.mean}, std={self.std})"


# Use it in a pipeline
torch.manual_seed(42)
x = torch.tensor([1.0, 2.0, 3.0])
noisy = AddGaussianNoise(std=0.1)(x)
print(f"Original: {x}")
print(f"Noisy:    {noisy}")

## 6. collate_fn for Variable-Length Data

By default, `DataLoader` stacks samples into a batch tensor. This fails when samples have **different sizes** (e.g., sentences of different lengths).

The `collate_fn` parameter lets you define custom batching logic.

In [ ]:
# Dataset with variable-length sequences
class VariableLengthDataset(Dataset):
    """Simulates sentences of different lengths."""

    def __init__(self, num_samples=20):
        torch.manual_seed(42)
        self.data = []
        for _ in range(num_samples):
            length = torch.randint(3, 10, (1,)).item()
            seq = torch.randn(length)  # variable length!
            label = torch.randint(0, 2, (1,)).item()
            self.data.append((seq, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


var_ds = VariableLengthDataset(num_samples=6)
for i in range(len(var_ds)):
    seq, label = var_ds[i]
    print(f"Sample {i}: length={len(seq)}, label={label}")

In [ ]:
# Default collate fails with variable-length data
try:
    loader_fail = DataLoader(var_ds, batch_size=3)
    batch = next(iter(loader_fail))
except RuntimeError as e:
    print(f"Default collate error: {e}")

In [ ]:
# Custom collate_fn: pad sequences to the same length
def pad_collate_fn(batch):
    """Pad variable-length sequences to the max length in the batch."""
    sequences, labels = zip(*batch)

    # Find max length in this batch
    max_len = max(len(s) for s in sequences)

    # Pad each sequence
    padded = []
    lengths = []
    for s in sequences:
        pad_amount = max_len - len(s)
        padded_seq = torch.cat([s, torch.zeros(pad_amount)])
        padded.append(padded_seq)
        lengths.append(len(s))

    # Stack into batch tensors
    padded_batch = torch.stack(padded)          # (batch, max_len)
    labels_batch = torch.tensor(labels)          # (batch,)
    lengths_batch = torch.tensor(lengths)        # (batch,)

    return padded_batch, labels_batch, lengths_batch


# Use custom collate_fn
loader_padded = DataLoader(var_ds, batch_size=3, collate_fn=pad_collate_fn)

for batch_idx, (seqs, labels, lengths) in enumerate(loader_padded):
    print(f"Batch {batch_idx}:")
    print(f"  Padded sequences shape: {seqs.shape}")
    print(f"  Labels: {labels}")
    print(f"  Original lengths: {lengths}")
    print()

## 7. random_split for Train/Val/Test

`torch.utils.data.random_split` splits a dataset into non-overlapping subsets.

In [ ]:
# Create a full dataset
torch.manual_seed(42)
full_X = torch.randn(1000, 10)
full_y = torch.randint(0, 5, (1000,))
full_ds = TensorDataset(full_X, full_y)

# Split: 70% train, 15% val, 15% test
n_total = len(full_ds)
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val  # ensures no samples are lost

train_ds, val_ds, test_ds = random_split(
    full_ds,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42),  # reproducible split
)

print(f"Total:  {n_total}")
print(f"Train:  {len(train_ds)} ({len(train_ds)/n_total:.0%})")
print(f"Val:    {len(val_ds)} ({len(val_ds)/n_total:.0%})")
print(f"Test:   {len(test_ds)} ({len(test_ds)/n_total:.0%})")

In [ ]:
# Create DataLoaders for each split
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Quick sanity check: grab one batch
X_b, y_b = next(iter(train_loader))
print(f"\nFirst train batch: X={X_b.shape}, y={y_b.shape}")

## 8. Exercises

### Exercise 1: Build a Dataset for CSV-Like Data

Create a `CSVDataset` class that:
1. Takes a list of dictionaries (simulating rows from a CSV file)
2. Extracts numerical features and a label column
3. Returns `(features_tensor, label_tensor)` from `__getitem__`

Test it with the sample data below.

In [ ]:
# Sample CSV-like data
csv_data = [
    {"height": 170.0, "weight": 65.0, "age": 30, "label": 0},
    {"height": 180.0, "weight": 80.0, "age": 25, "label": 1},
    {"height": 160.0, "weight": 55.0, "age": 35, "label": 0},
    {"height": 175.0, "weight": 72.0, "age": 28, "label": 1},
    {"height": 165.0, "weight": 60.0, "age": 40, "label": 0},
    {"height": 185.0, "weight": 90.0, "age": 22, "label": 1},
    {"height": 155.0, "weight": 50.0, "age": 45, "label": 0},
    {"height": 190.0, "weight": 85.0, "age": 33, "label": 1},
]

feature_columns = ["height", "weight", "age"]
label_column = "label"

# TODO: Implement CSVDataset

# class CSVDataset(Dataset):
#     def __init__(self, data, feature_cols, label_col):
#         ...
#
#     def __len__(self):
#         ...
#
#     def __getitem__(self, idx):
#         ...


# Test:
# ds = CSVDataset(csv_data, feature_columns, label_column)
# print(f"Size: {len(ds)}")
# x, y = ds[0]
# print(f"Features: {x}, Label: {y}")

# loader = DataLoader(ds, batch_size=4, shuffle=True)
# for X_b, y_b in loader:
#     print(f"Batch: X={X_b.shape}, y={y_b.shape}")

### Exercise 2: DataLoader Exploration

Using `TensorDataset` and `DataLoader`, create a dataset of 100 samples. Then:
1. Create a DataLoader with `batch_size=15` and `drop_last=False` -- how many batches? What is the size of the last batch?
2. Create a DataLoader with `batch_size=15` and `drop_last=True` -- how many batches now?

In [ ]:
# TODO: Explore DataLoader behaviour

# torch.manual_seed(42)
# X = torch.randn(100, 5)
# y = torch.randint(0, 2, (100,))
# ds = TensorDataset(X, y)

# 1. drop_last=False
# loader1 = DataLoader(ds, batch_size=15, drop_last=False)
# ...

# 2. drop_last=True
# loader2 = DataLoader(ds, batch_size=15, drop_last=True)
# ...

### Solutions

In [ ]:
# Solution 1: CSVDataset
class CSVDataset(Dataset):
    """Dataset that wraps CSV-like data (list of dicts)."""

    def __init__(self, data, feature_cols, label_col):
        # Extract features and labels from list of dicts
        features = [[row[col] for col in feature_cols] for row in data]
        labels = [row[label_col] for row in data]

        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# Test
ds = CSVDataset(csv_data, feature_columns, label_column)
print(f"Dataset size: {len(ds)}")

x, y = ds[0]
print(f"Features: {x}, Label: {y}")

loader = DataLoader(ds, batch_size=4, shuffle=True)
for X_b, y_b in loader:
    print(f"Batch: X shape={X_b.shape}, y shape={y_b.shape}")

In [ ]:
# Solution 2: DataLoader exploration
torch.manual_seed(42)
X = torch.randn(100, 5)
y = torch.randint(0, 2, (100,))
ds = TensorDataset(X, y)

# drop_last=False
loader1 = DataLoader(ds, batch_size=15, drop_last=False)
print(f"drop_last=False: {len(loader1)} batches")
for i, (xb, yb) in enumerate(loader1):
    if i == len(loader1) - 1:
        print(f"  Last batch size: {xb.shape[0]} (100 % 15 = {100 % 15})")

# drop_last=True
loader2 = DataLoader(ds, batch_size=15, drop_last=True)
print(f"\ndrop_last=True:  {len(loader2)} batches")
print(f"  All batches have exactly 15 samples")
print(f"  {100 - 15 * len(loader2)} samples are dropped")

## 9. Common Mistakes & Debugging Tips

1. **Not converting NumPy arrays to tensors in `__init__`**: Converting inside `__getitem__` is slow because it runs on every access. Convert once in `__init__`.

2. **Wrong dtype for labels**: `nn.CrossEntropyLoss` expects `torch.long` labels. Use `dtype=torch.long` when creating label tensors.

3. **Setting `num_workers > 0` on Windows without `if __name__ == '__main__'`**: On Windows, multi-process data loading requires the main script to be guarded. Use `num_workers=0` in notebooks.

4. **Forgetting `shuffle=True` for training**: Without shuffling, the model may learn the order of samples instead of their features.

5. **Using `shuffle=True` for validation/test**: Shuffling is not needed (and adds overhead) for evaluation. Use `shuffle=False`.

6. **Not setting a random seed in `random_split`**: Without a seed, you get a different split every time. Pass `generator=torch.Generator().manual_seed(42)`.

7. **Applying data augmentation during validation/test**: Augmentation transforms (flip, rotate, crop) should only be applied during training. Use separate transform pipelines.

8. **Variable-length data crashing DataLoader**: The default `collate_fn` tries to stack all samples, which fails if they have different shapes. Provide a custom `collate_fn` that pads or truncates.